In [ ]:
!pip install torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ======================================
# 1) LOAD DATA
# ======================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_data = datasets.MNIST(
    root='./data',
    train=False,
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

# ======================================
# 2) MODEL
# ======================================

model = nn.Sequential(

    nn.Flatten(),

    nn.Linear(28 * 28, 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 10)

)

# ======================================
# 3) LOSS & OPTIMIZER
# ======================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ======================================
# 4) TRAINING
# ======================================
model.train()
epochs = 5

train_losses = []
train_accuracy = []

for epoch in range(epochs):

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Forward
        outputs = model(images)

        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        # Accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    accuracy = correct / total

    train_losses.append(total_loss)
    train_accuracy.append(accuracy)

    print(f"Epoch {epoch+1}")
    print(f"Loss = {total_loss:.4f}")
    print(f"Accuracy = {accuracy:.4f}")
    print("-" * 40)

# ======================================
# 5) TESTING
# ======================================
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

test_accuracy = correct / total

print(f"\nTest Accuracy = {test_accuracy:.4f}")

# ======================================
# 6) VISUALIZATION
# ======================================

plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

plt.plot(train_accuracy)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()